# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and its Croissant schema. All entities - such as record sets, fields, and columns - are referenced by their `@id`, ensuring precise correspondence with the dataset schema.

### Dataset Source
The dataset is loaded using the Croissant schema at the following URL:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and discover available record sets and fields using `mlcroissant`. This also prints the dataset's title and description for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description (accessed as object attributes, not like a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
# Print the citation (if available)
if hasattr(dataset.metadata, 'cite_as') and dataset.metadata.cite_as:
    print(f"Cite as: {dataset.metadata.cite_as}")

## 2. Data Overview
List and explore available record sets with their fields and columns. All entities are reported using their `@id`.

In [ ]:
# List all record sets
print('Available record sets (@id):')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name','(no name)')}")

# For each record set, print its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print('  Fields:')
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")  # Sometimes only string
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print('  Columns:')
        for c in columns:
            if isinstance(c, dict) and '@id' in c:
                print(f"    - {c['@id']}")
            else:
                print(f"    - {c}")

## 3. Data Extraction
Load table-like data from the main record set(s) into pandas DataFrames for exploration. Here, each record set is referenced by its `@id`.

_Note: If the schema provides multiple record sets, all are loaded. In most cases, the main data table is the first or largest record set._

In [ ]:
# List the record set @ids to extract data from
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Read data for each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(1))

# Use the main record set for further examples (first in list)
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]
print(f"\nSelected main record set for analysis: {main_record_set_id}\n")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
This section shows basic filtering, normalization, and grouping of data using the main record set and references all fields by their `@id`.

Examples:
- Filtering by a numeric field such as molecular weight (often named by `@id` like `cr:field_molecular_weight`)
- Normalizing a numeric column
- Grouping by a categorical field (e.g., "description" or sample type column by `@id`)

_Replace the field `@id` values with those shown in your data overview if exploring a different column._

In [ ]:
# Identify numeric and group fields by their @id from the earlier overview.
# Example @id(s) guessed from a typical protein mass spec dataset:
# Let's try common options (please adapt if fields differ in your output):

import numpy as np

df = main_df  # Shortcut for main record set

# Example: Find any fields containing 'weight', 'MW', or similar for numeric analysis
candidate_numeric_fields = [col for col in df.columns if any(w in col.lower() for w in ['weight', 'mw','abundance','count','peptide','coverage','quant'])]
print('Numeric field candidates:', candidate_numeric_fields)

# Pick the first such field as the numeric field (override if needed)
numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else df.select_dtypes(include=[np.number]).columns[0]
print(f"Selected numeric field: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Example: Group by a description or categorical field
candidate_group_fields = [col for col in df.columns if any(gf in col.lower() for gf in ['description','type','sample','accession','group'])]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print('No group/categorical field found by @id for grouping example.')

## 5. Visualization
Create basic visualizations, such as a histogram of a numeric field or boxplot of normalized values, to explore data distributions using the field `@id`.

> Note: Adjust the field names according to your record set's columns and `@id`s as shown previously.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Histogram of {numeric_field_id}")
plt.show()

# If a group field exists, show comparison
if candidate_group_fields:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[candidate_group_fields[0]], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {candidate_group_fields[0]}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore, filter, and visualize the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all data elements by their `@id` in line with the Croissant schema. This approach ensures reproducible and schema-aware analytics for FAIR biomedical data sharing.

- **Key observations:**
    - Main record set(s) and fields identified by their `@id` help standardize programmatic data access.
    - Numeric columns such as protein abundance or molecular weight can be filtered and normalized for downstream analysis.
    - Grouping and visualization help identify meaningful patterns in the data.

Remember to reference the Croissant schema for exact field and record set IDs when extending your analyses.
